# 02 · 线性回归：从一条直线到过拟合

回归是预测**连续数值**。这个 notebook 讲三件事：

1. 怎么拟合一条线，怎么看它好不好
2. 特征标准化为什么重要
3. **过拟合长什么样**——这是机器学习最核心的坑

In [ ]:
import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "WenQuanYi Zen Hei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("中文字体:", matplotlib.rcParams["font.sans-serif"][0])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X, y = housing.data, housing.target

print("样本数:", X.shape[0], "特征数:", X.shape[1])
print("特征:", housing.feature_names)
print(f"房价中位数 {np.median(y):.2f} (单位: 10万美元)")

这份数据第一次运行会下载约 500KB。8 个特征都是数值型的，很干净——**真实数据不会这么友好**，下一节会讲怎么处理脏数据。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

lr = LinearRegression().fit(X_train, y_train)
pred = lr.predict(X_test)

print(f"MAE  {mean_absolute_error(y_test, pred):.3f}")
print(f"RMSE {root_mean_squared_error(y_test, pred):.3f}")
print(f"R²   {r2_score(y_test, pred):.3f}")

三个指标分别是什么：

- **MAE**：平均差多少。直观，但大误差和小误差一视同仁
- **RMSE**：差的平方再开根。大误差被放大，所以对「偶尔错得离谱」更敏感
- **R²**：模型解释了数据的多少比例。1 是完美，0 等于「只会猜平均值」，**负数说明比猜平均值还差**

R² 约 0.6 意味着：房价的大部分波动这套线性模型抓不到。这很正常，地理位置被压缩成两个数字，信息损失太大。

## 标准化：让特征站在同一起跑线

这份数据里 `MedInc`（收入）是几万，`HouseAge`（房龄）是几十。量纲差三个数量级。

对线性回归影响不大，但对**正则化、KNN、SVM、神经网络**影响巨大——它们靠距离或梯度工作，大数值特征会直接支配结果。

关键点：`fit` 只能用训练集，`transform` 两边都用。**用测试集算均值就作弊了。**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Pipeline 把「标准化 + 模型」打包成一个对象，避免手写两遍 fit/transform 时出错
pipe = make_pipeline(StandardScaler(), LinearRegression())
pipe.fit(X_train, y_train)
print(f"标准化后 R²: {pipe.score(X_test, y_test):.3f}")

# 对比几个非线性模型
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

for name, m in [
    ("线性回归", LinearRegression()),
    ("随机森林", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ("梯度提升", GradientBoostingRegressor(random_state=42)),
]:
    p = make_pipeline(StandardScaler(), m).fit(X_train, y_train)
    print(f"{name:<8} R² = {p.score(X_test, y_test):.3f}")

树模型不用标准化也能到 0.8，因为它是**按阈值切分**的，不关心数值的绝对大小——只看排序。

## 过拟合演示

用一维数据看得最清楚。同一批点，用 1 次、3 次、15 次多项式去拟合：

- 1 次：太简单，欠拟合（underfitting）
- 3 次：刚好
- 15 次：穿过每个点，但在点之间疯狂摆动——过拟合（overfitting）

**过拟合的特征是：训练误差很低，测试误差很高。**

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

rng = np.random.RandomState(0)
x = np.sort(rng.rand(25) * 10)
y_true = np.sin(x)
y_noisy = y_true + rng.normal(0, 0.2, size=x.shape)
x_plot = np.linspace(0, 10, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, deg in zip(axes, [1, 3, 15]):
    p = Pipeline([("poly", PolynomialFeatures(deg)), ("lr", LinearRegression())])
    p.fit(x.reshape(-1, 1), y_noisy)
    train_err = mean_squared_error(y_noisy, p.predict(x.reshape(-1, 1)))
    ax.scatter(x, y_noisy, s=18, label="观测数据")
    ax.plot(x_plot, p.predict(x_plot.reshape(-1, 1)), color="crimson", label=f"{deg} 次多项式")
    ax.plot(x_plot, np.sin(x_plot), "k--", lw=1, alpha=0.6, label="真实规律")
    ax.set_title(f"{deg} 次 · 训练 MSE={train_err:.4f}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 正则化：用惩罚压住过拟合

Ridge（L2）给系数加平方惩罚，Lasso（L1）加绝对值惩罚。共同点：**系数越大罚得越重**，逼模型别那么激进。

`alpha` 是惩罚力度。太小没用，太大直接压成平线——**这又是一个超参数**。

> 运行时如果看到 `LinAlgWarning: An ill-conditioned matrix detected`，不是 bug：alpha 极小时 15 次多项式的设计矩阵接近奇异（列与列几乎线性相关），数值求解会警告。把 alpha 调大一位数警告就消失——这个警告本身就是「模型病态」的信号，值得认识。

In [ ]:
from sklearn.linear_model import Ridge

for alpha in [0, 1e-6, 1e-3, 0.1, 10]:
    p = Pipeline([
        ("poly", PolynomialFeatures(15)),
        ("ridge", Ridge(alpha=alpha)),
    ]).fit(x.reshape(-1, 1), y_noisy)
    # 用真实规律算误差，检验它有没有学到 sin 而不是噪声
    err = mean_squared_error(np.sin(x_plot), p.predict(x_plot.reshape(-1, 1)))
    print(f"alpha={alpha:<8} 对真实规律的 MSE = {err:.5f}")

## 留给你动手的

1. 把 `alpha` 从 `1e-6` 扫到 `100`（用 `np.logspace(-6, 2, 30)`），画出 MSE 曲线，找出最好的那个点。
2. 样本量从 25 改到 200，重新跑过拟合那节——**为什么数据多了就不容易过拟合？**
3. 把第三张图的 `PolynomialFeatures(15)` 换成 `PolynomialFeatures(15, include_bias=False)`，结果变了吗？
4. 试试 `Lasso`：它会把一部分系数压成**正好 0**，等于自动挑特征。这对高维数据很有用。